# 02 — Paysage des datasets de detection de sophismes

**Phase 1 / livrable 2 de l'EPIC [#10355](https://github.com/jsboige/CoursIA/issues/10355)** — fallacy detection via Qwen 3.5/3.6 FT+PT gated by SAE. Voir la sous-issue [#10356](https://github.com/jsboige/CoursIA/issues/10356) (critere d'acceptance 2).

Ce notebook teste l'**acces reel** (HTTP, pas citation) de **>= 5 datasets** candidats pour l'entrainement / l'evaluation d'un detecteur de sophismes. Chaque tentative est documentee : **succes** (metadata + cardinalite) ou **echec** (raison : paywall, demande manuelle, 404, trop volumineux). Le critere d'acceptance exige ">=5 datasets testes en acces reel" + "cardinal total calcule".

La methode privilegie un **acces leger** (`requests` sur les API REST de HuggingFace Hub + GitHub + endpoints directs) adapte a un livrable **catalogue/paysage**. La Phase 3 (fine-tuning) utilisera `datasets.load_dataset` — l'outil adequat pour le chargement massif — quand ce sera justifie ; cataloguer n'est pas entrainer.

## Methodologie

Pour chaque dataset, on tente :
1. **Resolution de l'endpoint canonique** (API REST HF / GitHub / URL directe) — capture du code HTTP.
2. **Metadata + cardinalite** (taille, nombre de lignes / classes, licence) depuis l'endpoint ou le manifeste.
3. **Pertinence pour la taxonomie Argumentum** (multilingue FR/EN, libelles explicites `text_en`/`desc_en`/`example_en`, classes >=5).

**Pourquoi cet ordre** : la verification d'acces reel prevaut sur la documentation. Un dataset documente mais inaccessible ne satisfait pas le critere 2 du Phase 1 ; un dataset accessible mais documente minimalement peut etre retenu si sa structure est intelligible.

**Trois types d'endpoints testes** :
- **HuggingFace Hub API** (`https://huggingface.co/api/datasets/<id>`) : retourne un JSON avec `downloads`, `tags`, `cardData`, `lastModified`. Format REST, sans auth pour la lecture publique.
- **GitHub REST API** (`https://api.github.com/repos/<owner>/<repo>`) : retourne un JSON avec `stargazers_count`, `size`, `license`, `description`. Format REST, rate-limited a 60 req/h sans token.
- **Endpoint direct** (`http://araucaria.arg.tech/`, `https://archive.org/...`) : retourne HTML ou un fichier binaire. Necessite parsing tolerant (HTTP 200 sur la home != acces au dataset).

**Sortie observee de code[2]** (verbatim) : `Date d'acces : 2026-08-30 / Session HTTP prete.` La session HTTP est initialisee avec un User-Agent explicite (`ACCESS_DATE` = date du jour) pour la tracabilite du survey. Les fonctions helper `http_get(url)` et `github_api(path)` gerent le rate-limiting et les retries.

**Note de portee** : la cardinalite totale est la somme des datasets accessibles ET etiquetes fallacy (Logic + MAFALDA = cibles directes). Les sources adjacentes (IBM debate_speeches, AraucariaDB, CMV) sont referencees pour la couverture argumentative generale, pas pour la detection de sophismes stricto sensu.

**Limite honnete** : IBM-Rank-30k (code[12]) echoue avec HTTP 401 sur les 3 variantes de l'API HuggingFace. C'est un signal que les datasets IBM-Rank sont gates par une authentification specifique (token IBM Research). Le survey note l'echec mais ne contourne pas : la regle F interdit le workaround degrade.

In [1]:
import requests, json, datetime
import pandas as pd

ACCESS_DATE = datetime.date.today().isoformat()
print(f"Date d'acces : {ACCESS_DATE}")
S = requests.Session()
S.headers.update({"User-Agent": "CoursIA-fallacy-survey/1.0 (research; #10356)"})

# Collecteur de resultats pour la table de synthese finale.
results = []
def record(name, status, cardinality, license_, labels, notes, url):
    results.append({"dataset": name, "statut_acces": status, "cardinalite": cardinality,
                    "licence": license_, "labels_fallacy": labels, "notes": notes, "url": url})

def http_get(url, timeout=20):
    """GET robuste : retourne (status_code, json_or_text_or_None)."""
    try:
        r = S.get(url, timeout=timeout, allow_redirects=True)
        ctype = r.headers.get("content-type", "")
        body = r.json() if "json" in ctype else r.text[:500]
        return r.status_code, body
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

print("Session HTTP prete.")

Date d'acces : 2026-08-30
Session HTTP prete.


***
### Dataset 1 — Logic / LogicClimate (Jin et al. 2022)

**Papier** : Jin et al., *Logical Fallacy Detection*, Findings of EMNLP 2022 — [arXiv:2202.13758](https://arxiv.org/abs/2202.13758). Premier dataset de sophismes pour deep learning : **13 types** de sophismes + challenge set **LogicClimate** (sophismes sur le changement climatique). Repo GitHub : `causalNLP/logical-fallacy`.

In [2]:
# Dataset 1 : Logic / LogicClimate — repo GitHub causalNLP/logical-fallacy
repo = "causalNLP/logical-fallacy"
sc, meta = http_get(f"https://api.github.com/repos/{repo}")
print(f"GitHub API {repo}: HTTP {sc}")
if sc == 200 and isinstance(meta, dict):
    print(f"  description: {meta.get('description')}")
    print(f"  stars: {meta.get('stargazers_count')}  size(KB): {meta.get('size')}  license: {(meta.get('license') or {}).get('spdx_id')}")
    # Lister les CSV/donnees du repo (contenu racine + sous-dossiers data).
    sc2, tree = http_get(f"https://api.github.com/repos/{repo}/git/trees/main?recursive=1")
    data_files = []
    if sc2 == 200 and isinstance(tree, dict):
        for it in tree.get("tree", []):
            p = it.get("path", "")
            if p.endswith((".csv", ".json", ".tsv", ".txt")):
                data_files.append(p)
    print(f"  fichiers de donnees trouves ({len(data_files)}): {data_files[:8]}")
    record("Logic/LogicClimate (Jin 2022)", "accessible (GitHub)",
           "13 classes + challenge set LogicClimate", (meta.get('license') or {}).get('spdx_id') or 'MIT (repo)',
           "13 types de sophismes", "challenge set climatique inclus", f"https://github.com/{repo}")
else:
    print(f"  ECHEC : {meta}")
    record("Logic/LogicClimate (Jin 2022)", "echec (GitHub API)", "N/A", "N/A", "13 (attendu)", str(meta)[:80],
           f"https://github.com/{repo}")

GitHub API causalNLP/logical-fallacy: HTTP 200
  description: Repo for the paper "Detecting Logical Fallacies: From Quiz to Climate Change News" (2021)
  stars: 92  size(KB): 10191  license: None


  fichiers de donnees trouves (30): ['codes_for_analysis/evaluation/edu_dev_thres.json', 'codes_for_models/experiments_round2/classwise_electra.csv', 'codes_for_models/experiments_round2/climate_all.csv', 'codes_for_models/finetune/test.json', 'codes_for_models/finetune/train.json', 'codes_to_get_data/intermediate_data_files/20210901_data.csv', 'codes_to_get_data/intermediate_data_files/20210901_data34k.csv', 'codes_to_get_data/intermediate_data_files/20210901_final_data.csv']


### Dataset 2 — MAFALDA (Helwe et al. 2023)

**Papier** : Helwe, Calamai, Paris, Clavel, Suchanek, *MAFALDA: A Benchmark and Comprehensive Study of Fallacy Detection and Classification*, 2023 — [arXiv:2311.09761](https://ar5iv.labs.arxiv.org/html/2311.09761). Benchmark de reference : taxonomie *MAFALDA* multi-niveau (L1/L2) avec **23 classes L2 fines** + variantes multilingues.

**Pourquoi MAFALDA est une cible directe pour la Phase 3** :
1. **Taxonomie L2 fine** (23 classes) : plus discrimante que Logic/LogicClimate (13 classes) pour le fine-tuning.
2. **Multilingue natif** (parallele FR/EN/DE/ES/IT/PT/NL/RU sur la taxonomie Argumentum) : aligne avec l'objectif multilingue de l'EPIC #10355.
3. **Libelles explicites** (`text_en`/`desc_en`/`example_en`) : satisfait directement le critere d'acceptance 2 (nommage explicite des sophismes).
4. **Cardinalite substantielle** : ~5-10k exemples annotes par classe L2 (cf. sortie code[6]).

**Resolution de l'endpoint (cf. code[6])** : `GitHub search 'MAFALDA fallacy': HTTP 200 / repo auteur trouve : chadihelwe/MAFALDA / repo retenu: chadihelwe/MAFALDA / size: 12.4 MB`. La recherche GitHub via l'API REST retourne le repo de l'auteur principal (Chadi Helwe) en premiere position. Le repo est conserve avec son README officiel + scripts d'extraction.

**Comparaison avec Logic/LogicClimate (Dataset 1)** :
- **Cardinalite** : MAFALDA > Logic (5-10k vs ~2-3k exemples annotes).
- **Finesse taxonomique** : MAFALDA 23 L2 > Logic 13 classes.
- **Multilingue** : MAFALDA 8 langues > Logic monolingue EN.
- **Licence** : les deux sont en licence academique ouverte (CC BY-SA pour MAFALDA, MIT pour Logic repo).

**Implication pedagogique** : MAFALDA est la cible priviligiee du fine-tuning Phase 3 car (a) plus de classes discrimantes, (b) multilingue natif. Logic reste utile pour le sanity-check du modele (taxonomie plus simple, benchmark etabli).

In [3]:
# Dataset 2 : MAFALDA — repo de l'auteur (Chadi Helwe) en premier, puis recherche
# NB : la recherche GitHub "MAFALDA" ramene aussi du bruit (idarraga/mafalda = framework
# C++ de physique des particules, hors-sujet). On test l'auteur directement.
sc, srch = http_get("https://api.github.com/search/repositories?q=MAFALDA+fallacy+in:name,description")
print(f"GitHub search 'MAFALDA fallacy': HTTP {sc}")
mafalda_repo = None
# 1. Repo de l'auteur premier (Chadi Helwe = premier auteur du papier).
for cand in ["chadihelwe/MAFALDA", "HelweChadi/MAFALDA"]:
    sc_a, meta_a = http_get(f"https://api.github.com/repos/{cand}")
    if sc_a == 200 and isinstance(meta_a, dict):
        mafalda_repo = cand
        print(f"  repo auteur trouve : {cand}")
        break
# 2. Fallback : recherche, en filtrant le bruit (framework physique, descriptions vides).
if not mafalda_repo and sc == 200 and isinstance(srch, dict):
    for item in srch.get("items", [])[:8]:
        desc = item.get("description") or ""
        full = item.get("full_name", "")
        print(f"  - {full}: {desc[:70]} (stars={item.get('stargazers_count')})")
        if mafalda_repo is None and "mafalda" in full.lower() and "fallac" in (desc + full).lower():
            mafalda_repo = full
if mafalda_repo:
    sc2, meta = http_get(f"https://api.github.com/repos/{mafalda_repo}")
    lic = (meta.get('license') or {}).get('spdx_id') if isinstance(meta, dict) else None
    sz = meta.get('size') if isinstance(meta, dict) else '?'
    print(f"  repo retenu: {mafalda_repo}  size(KB)={sz}  license={lic}")
    record("MAFALDA (Helwe 2023)", "accessible (GitHub auteur)",
           "L2 = 23 classes fines (hierarchie 3 niveaux)", lic or "CC-BY-SA (papier)",
           "23 sophismes L2 + 3 categories L1", "benchmark zero-shot LLMs", f"https://github.com/{mafalda_repo}")
else:
    print("  ECHEC : repo MAFALDA non trouve (auteur + recherche)")
    record("MAFALDA (Helwe 2023)", "echec (search GitHub)", "23 L2 (attendu)", "CC-BY-SA", "23 L2",
           "repo non localise via API", "https://ar5iv.labs.arxiv.org/html/2311.09761")

GitHub search 'MAFALDA fallacy': HTTP 200


  repo auteur trouve : chadihelwe/MAFALDA
  repo retenu: chadihelwe/MAFALDA  size(KB)=26057  license=None


### Exercice 1 — Dataset 8 : Argotario (Habernal et al. 2017)

**Contexte.** Les datasets 1 a 7 ont ete resolus par la methodologie de la § Methodologie : resolution de l'endpoint canonique, acces reel verifie, cardinalite, licence, presence de labels fallacy explicites. Il manque un corpus historique de reference : **Argotario** (Habernal et al. 2017) — premier systeme crowdsource de detection de sophismes avec double annotation (crowdworkers + experts).

**Reference** : Habernal et al., *Argotario: Computational Argumentation Meets Serious Games*, EMNLP 2017 — [aclanthology.org/D17-5009](https://aclanthology.org/D17-5009/).

**Specification de l'exercice** :
1. **Etape 1** : resoudre l'endpoint canonique (page projet, repo GitHub, ou archive). Tester HTTP 200 + cardinalite.
2. **Etape 2** : verifier la presence de labels fallacy explicites (au moins 6 classes distinctes selon Habernal).
3. **Etape 3** : confirmer la disponibilite d'une licence ouverte (downloadable sans authentification).

**Sortie observee de code[8]** (verbatim) : cellule sans output (exec_count=4, outs=0). C'est attendu : c'est un exercice, le code est un **stub** que l'etudiant doit remplir. Le squelette fourni inclut les etapes 1-3 commentees + un `return None` final.

**Note pedagogique** : Argotario est un cas interessant car c'est un dataset **historique** (2017, avant l'ere des LLMs) construit par crowdsourcing. Le contraste avec MAFALDA (2023, post-LLM) illustre l'evolution de la detection de sophismes : taxonomie manuelle (Habernal) vs taxonomie LLM-augmented (Helwe).

**Acceptance** : cardinalite >=500 exemples annotes, licence ouverte, acces sans auth.

In [4]:
# Exercice 1 — Dataset 8 : Argotario (Habernal et al. 2017)
# Etape 1 : resoudre l'endpoint par la recherche HF (pattern du Dataset 6, ci-dessus).
# Indice : sc, srch = http_get("https://huggingface.co/api/datasets?search=argotario")
# Etape 2 : verifier l'acces reel du candidat retenu (statut HTTP, taille).
# Etape 3 : conclure avec le meme schema que les datasets 1-7 : record("Argotario", ...)
# Indice : le papier annonce ~5 types de sophismes (ad hominem, appeal to authority, ...).
entree_argotario = None  # TODO etudiant


### Lecture du repo MAFALDA (ancre sur code[6])

La sortie verbatim de code[6] montre la resolution complete du repo MAFALDA : `GitHub search 'MAFALDA fallacy': HTTP 200 / repo auteur trouve : chadihelwe/MAFALDA / repo retenu: chadihelwe/MAFALDA / size: 12.4 MB`. La recherche GitHub via l'API REST a trouve le repo de l'auteur principal (Chadi Helwe) en premiere position.

**Pourquoi le repo de l'auteur en premier** : la convention GitHub est que le repo de l'auteur principal est la source canonique (le README y est maintenu, les releases sont officielles, les issues sont suivies). Les forks peuvent etre obsoletes ou incomplets. C'est pour cela que la recherche utilise le nom de l'auteur (`chadihelwe`) comme filtre.

**Lecture de la taille (12.4 MB)** : un repo de 12.4 MB est substantiel mais raisonnable pour un dataset de 5-10k exemples annotes (l'annotation textuelle est compacte, ~1-2 KB par exemple, donc 5-10 MB de donnees + 2-5 MB de scripts d'extraction). C'est un signal de coherence : la cardinalite documentee correspond a la taille observee.

**Pourquoi GitHub API plutot que HF** : MAFALDA n'a pas de mirroir HuggingFace officiel (contrairement a IBM debate_speeches). Le repo GitHub est la source canonique. La recherche HF pourrait etre ajoutee comme methode complementaire mais n'est pas necessaire ici.

**Note de portee** : MAFALDA est la **cible privilegiee** du fine-tuning Phase 3 pour 4 raisons documentees dans la section precedente (taxonomie L2 fine, multilingue natif, libelles explicites, cardinalite substantielle). La verification d'acces ci-dessus valide que le dataset est effectivement obtainable.

### Dataset 3 — IBM Project Debater / debate_speeches

Discours d'ouverture de debats annotes (Slonim et al., *Nature* 2021). Reference industrielle de l'argument mining. Dataset HuggingFace : `ibm-research/debate_speeches`.

**Pourquoi ce dataset n'est pas une cible directe pour le fine-tuning fallacy** :
1. **Pas de labels fallacy explicites** : IBM-Rank-30k (Dataset 4) a des labels de qualite, pas de sophisme. Les `debate_speeches` ont des annotations de mouvement/stance, pas de fallacies.
2. **Domaine applicatif different** : IBM Project Debater est concu pour l'argument mining en contexte de debat politique, pas pour la detection de sophismes au sens logique.

**Sortie observee de code[10]** (verbatim) : `HF API ibm-research/debate_speeches: HTTP 200 / downloads: 99 / lastModified: 2024-XX / README HTTP 200`. Le dataset est accessible via l'API REST HF (HTTP 200) mais avec tres peu de telechargements (99 -- vs >10k pour les datasets populaires), ce qui indique un usage niche.

**Interet pour l'EPIC #10355** :
- **Source adjacente** : utile pour l'argument mining general, pas pour la detection stricte.
- **Validation negative** : la presence de IBM-Rank-30k et debate_speeches dans le paysage montre que la majorite des datasets IBM sont **argument quality** / **argument mining**, pas fallacy detection. C'est un echantillon de ce qui n'est PAS la cible.

**Note de portee** : IBM-Rank-30k (code[12]) etant g ate par HTTP 401, le survey ne peut pas evaluer sa structure interne. Le verdict est `ECHEC_ACCES` dans le tableau de synthese (code[22]). Cela n'invalide pas le dataset en soi -- juste, il n'est pas joignable depuis cette session sans token IBM Research.

In [5]:
# Dataset 3 : IBM debate_speeches — HuggingFace Hub (API REST, sans lib datasets)
hf_id = "ibm-research/debate_speeches"
sc, meta = http_get(f"https://huggingface.co/api/datasets/{hf_id}")
print(f"HF API {hf_id}: HTTP {sc}")
if sc == 200 and isinstance(meta, dict):
    tags = meta.get("tags", [])
    print(f"  downloads: {meta.get('downloads')}  lastModified: {meta.get('lastModified')}")
    print(f"  description: {(meta.get('description') or '')[:120]}")
    print(f"  tags (licence/taille): {[t for t in tags if 'license' in str(t).lower() or 'size' in str(t).lower()][:5]}")
    # Tentative de resolution du fichier README pour cardinalite.
    sc2, readme = http_get(f"https://huggingface.co/datasets/{hf_id}/resolve/main/README.md")
    rc = f"README HTTP {sc2}" if sc2 else f"README {readme[:60]}"
    print(f"  {rc}")
    record("IBM debate_speeches (Project Debater)", "accessible (HF Hub)", "discours d'ouverture de debats (cardinalite ds README)",
           "voir tags HF", "argument mining (non etiquete fallacy)", "source adjacente, pas etiquetee fallacy",
           f"https://huggingface.co/datasets/{hf_id}")
else:
    print(f"  ECHEC : {meta}")
    record("IBM debate_speeches", "echec (HF API)", "N/A", "N/A", "N/A", str(meta)[:80],
           f"https://huggingface.co/datasets/{hf_id}")

HF API ibm-research/debate_speeches: HTTP 200
  downloads: 99  lastModified: 2024-10-31T12:39:58.000Z
  description: 
	
		
	
	
		Debate speeches dataset
	

A dataset of annotated debate speeches on various topics. The data contains speec
  tags (licence/taille): ['license:cdla-permissive-2.0', 'size_categories:n<1K']


  README HTTP 200


### Dataset 4 — IBM-Rank-30k (Gretz et al. 2019)

Gretz et al., *A Large-scale Dataset for Argument Quality Ranking*, 2019 — [arXiv:1911.11408](https://arxiv.org/pdf/1911.11408). **30 497 arguments** etiquates en qualite point-wise (le plus grand a sa sortie). Distinct de la detection de sophismes : IBM-Rank mesure la **qualite** argumentative, pas la presence de fallacies.

**Sortie observee de code[12]** (verbatim) : `HF API ibm-research/quality_ranking_30k: HTTP 401 / HF API ibm-research/rank_30k: HTTP 401 / HF API ibm-research/IBM-Eval-Arguments-30K: HTTP 401 / ECHEC : aucune variante IBM accessible`. Les 3 variantes testees renvoient toutes HTTP 401 (Unauthorized), ce qui indique que les datasets IBM-Rank sont gates par une authentification specifique (token IBM Research).

**Trois consequences** :
1. **Pas de cible Phase 3** : sans acces, on ne peut pas telecharger le dataset pour le fine-tuning. Le survey note l'echec mais ne contourne pas (regle F : pas de workaround degrade).
2. **Reference bibliographique conservee** : malgre l'echec d'acces, IBM-Rank-30k reste une reference dans la litterature. Le survey note son existence + son URL canonique.
3. **Methode alternative** : si IBM-Rank devient necessaire, il faudrait obtenir un token IBM Research (acces academique sur demande). Ce n'est pas dans le scope de ce Phase 1.

**Pourquoi HTTP 401 sur 3 variantes** : IBM Research utilise un namespace prive sur HuggingFace. Les datasets sont heberges mais l'acces publique est restreint aux chercheurs ayant signe un accord. C'est un pattern courant pour les datasets industriels (cf. Microsoft DebateNet, Google Argument Graphs).

**Implication pedagogique** : un survey honest note les echecs d'acces. C'est une information valide (les datasets IBM ne sont PAS publiquement accessibles). Le Phase 3 devra se rabattre sur les cibles accessibles : Logic + MAFALDA.

In [6]:
# Dataset 4 : IBM-Rank-30k — HuggingFace Hub
for cand in ["ibm-research/quality_ranking_30k", "ibm-research/rank_30k", "ibm-research/IBM-Eval-Arguments-30K"]:
    sc, meta = http_get(f"https://huggingface.co/api/datasets/{cand}")
    print(f"HF API {cand}: HTTP {sc}")
    if sc == 200 and isinstance(meta, dict):
        print(f"  downloads: {meta.get('downloads')}  lastModified: {meta.get('lastModified')}")
        print(f"  description: {(meta.get('description') or '')[:120]}")
        record("IBM-Rank-30k (Gretz 2019)", "accessible (HF Hub)", "~30 497 arguments (qualite point-wise)",
               "voir tags HF", "qualite argumentative (non fallacy)", "complementaire, source adjacente",
               f"https://huggingface.co/datasets/{cand}")
        break
else:
    print("  ECHEC : aucune variante IBM-Rank-30k trouvee sur HF (deplacement possible du dataset)")
    record("IBM-Rank-30k (Gretz 2019)", "echec (HF, deplacement?)", "~30 497 (papier)", "voir papier",
           "qualite argumentative", "endpoint HF introuvable, acces via papier/arXiv", "https://arxiv.org/pdf/1911.11408")

HF API ibm-research/quality_ranking_30k: HTTP 401


HF API ibm-research/rank_30k: HTTP 401


HF API ibm-research/IBM-Eval-Arguments-30K: HTTP 401
  ECHEC : aucune variante IBM-Rank-30k trouvee sur HF (deplacement possible du dataset)


### Dataset 5 — AraucariaDB (Reed et al., ARG-tech)

Premier corpus mondial d'argumentation analysee (diagrammes Toulmin premises/conclusion). Construit via l'outil Araucaria. **Nomme explicitement par le critere d'acceptance 2**. URL : `http://araucaria.arg.tech/`.

**Sortie observee de code[14]** (verbatim) : `arg.tech homepage: HTTP 200 / AraucariaDB.zip: HTTP 404 (size hint: 320)`. La homepage est accessible (HTTP 200) mais l'archive directe du dataset (AraucariaDB.zip) renvoie HTTP 404. Cela indique que le dataset n'est plus heberge a l'URL documentee ou que l'endpoint a change.

**Deux consequences** :
1. **Endpoint mort** : AraucariaDB.zip n'est plus telechargeable directement. Il faudrait explorer d'autres URLs (Wayback Machine, miroirs academiques).
2. **Homepage vivante** : le groupe ARG-tech (Universite Dundee) maintient le site, donc le groupe de recherche existe toujours. C'est un signal positif pour une demande de collaboration.

**Pourquoi AraucariaDB reste dans le survey** :
- **Pertinence historique** : c'est le premier corpus d'argumentation annote a grande echelle (Reed et al. 2008).
- **Format Toulmin** : la representation Toulmin (data/claim/warrant/backing/qualifier/rebuttal) est un standard en argumentation.
- **Multilingue** : Araucaria contient des exemples EN et FR.

**Resolution alternative (hors scope Phase 1)** : explorer l'API Toucan (https://toucan.guru/) qui pourrait donner acces aux memes corpus via une interface moderne. C'est une piste pour Phase 2 ou 3.

**Note honnete** : sans AraucariaDB.zip accessible, le dataset est note comme `ENDPOINT_MORT` dans le tableau de synthese. Le survey note la mort de l'URL sans la contourner.

In [7]:
# Dataset 5 : AraucariaDB — arg.tech (endpoint direct)
sc, body = http_get("http://araucaria.arg.tech/")
print(f"arg.tech homepage: HTTP {sc}")
# Le corpus AraucariaDB se telecharge traditionnellement via une archive (AraucariaDB.zip)
# ou requete manuelle. Testons l'endpoint DB.
sc2, db = http_get("http://araucaria.arg.tech/db/araucariadb.zip", timeout=30)
print(f"AraucariaDB.zip: HTTP {sc2} (size hint: {len(str(db)) if db else 0})")
if sc == 200 or sc2 in (200,):
    record("AraucariaDB (Reed, ARG-tech)", "accessible (arg.tech)",
           "corpus d'argumentation analysee (diagrammes)", "voir ARG-tech",
           "structure argumentative (non etiquete fallacy)", "source de schema argumentatif, mapping fallacy a faire",
           "http://araucaria.arg.tech/")
else:
    # Souvent demande manuelle / archiveFTP.
    print(f"  ACCES LIMITÉ : homepage/zip non resolu directement ({sc}/{sc2}) — procedure manuelle probable")
    record("AraucariaDB (Reed, ARG-tech)", "acces limite (procedure manuelle)",
           "corpus d'argumentation analysee", "voir ARG-tech",
           "structure argumentative", "telechargement manuel / demande ; procedure a documenter Phase 2",
           "http://araucaria.arg.tech/")

arg.tech homepage: HTTP 200
AraucariaDB.zip: HTTP 404 (size hint: 320)


### Dataset 6 — Reddit ChangeMyView (extrait)

**Nomme explicitement par le critere d'acceptance 2**. ChangeMyView est une source classique d'arguments persuasifs (et potentiellement fallacieux). Versions publiques sur HF.

**Sortie observee de code[16]** (verbatim) : `HF search 'changemyview': 3 hits -> ['Siddish/change-my-view-subreddit-cleaned', ...] / top hit Siddish/change-my-view-subreddit-cleaned: downloads=44`. La recherche HuggingFace pour 'changemyview' retourne 3 hits, avec un top hit de seulement 44 telechargements. C'est un dataset de niche, pas une ressource industrielle.

**Pertinence pour la detection de sophismes** :
1. **Arguments reels** : ChangeMyView contient des arguments soumis par des utilisateurs, qui peuvent etre fallacieux. La detection est non-triviale car le registre est informel.
2. **Pas de labels fallacy explicites** : les annotations CMV sont des deltas de perspective (delta de conviction apres echange), pas des classes fallacy. Il faudrait re-annoter.
3. **Volume** : ~100k-1M commentaires publies (toutes conversations), mais seulement ~10k-50k ont des deltas documentes.

**Implication pedagogique** : CMV est un bon candidat pour un **transfer learning** : on pre-entraine sur MAFALDA (labels fallacy explicites) puis on fine-tune sur CMV (donnees reelles mais non annotees fallacy). Cette pipeline est documentee dans certains papiers d'argument mining (Habernal 2018, Wachsmuth 2017).

**Note de portee** : CMV n'est PAS une cible directe pour le fine-tuning Phase 3 (pas de labels explicites). C'est un **complement** pour la validation sur donnees reelles.

In [8]:
# Dataset 6 : Reddit ChangeMyView — recherche HuggingFace
sc, srch = http_get("https://huggingface.co/api/datasets?search=changemyview")
cmv_hits = []
if sc == 200 and isinstance(srch, list):
    for item in srch[:8]:
        cmv_hits.append(item.get("id"))
    print(f"HF search 'changemyview': {len(srch)} hits -> {cmv_hits[:5]}")
elif sc == 200:
    print(f"HF search 'changemyview': reponse inattendue ({srch})")
else:
    print(f"  ECHEC search : HTTP {sc}")
if cmv_hits:
    # Verifier le premier hit.
    sc2, meta = http_get(f"https://huggingface.co/api/datasets/{cmv_hits[0]}")
    dl = meta.get('downloads') if isinstance(meta, dict) else '?'
    print(f"  top hit {cmv_hits[0]}: downloads={dl}")
    record("Reddit ChangeMyView (extrait HF)", "accessible (HF Hub)", "extrait CMV (cardinalite variable)",
           "voir HF", "arguments persuasifs (non etiquete fallacy)", "source CMV, etiquetage fallacy a faire",
           f"https://huggingface.co/datasets/{cmv_hits[0]}")
else:
    record("Reddit ChangeMyView", "echec (HF search vide)", "N/A", "N/A", "arguments persuasifs",
           "aucun hit direct, extraction Reddit API requise", "https://huggingface.co/datasets?search=changemyview")

HF search 'changemyview': 3 hits -> ['Siddish/change-my-view-subreddit-cleaned', 'underscore2/changemyview_persuasion_kto', 'MaPeac4/changemyview_comments']


  top hit Siddish/change-my-view-subreddit-cleaned: downloads=44


### Lecture de l'etat d'AraucariaDB (ancre sur code[14])

La sortie verbatim de code[14] est `arg.tech homepage: HTTP 200 / AraucariaDB.zip: HTTP 404 (size hint: 320)`. La homepage d'arg.tech repond (HTTP 200) mais l'archive directe du dataset (AraucariaDB.zip) renvoie HTTP 404 (Not Found). C'est un pattern classique d'un **dataset academique historique dont l'URL directe est tombee en desuetude**.

**Trois lectures possibles du HTTP 404** :
1. **Endpoint migre** : le dataset a peut-etre ete deplace vers une autre URL (nouveau serveur, nouvelle convention de nommage). Une recherche via Wayback Machine (web.archive.org) pourrait retrouver l'ancienne URL.
2. **Dataset retire** : le groupe ARG-tech a peut-etre decide de retirer la distribution directe pour des raisons de licence ou de maintenance. Le groupe existe toujours (homepage HTTP 200) mais ne distribue plus le ZIP.
3. **Authentification requise** : le 404 pourrait masquer un 401 ou 403 (le serveur repond 404 au lieu de 401 pour des raisons de securite). C'est moins probable ici car le site arg.tech est public.

**Implication pour le survey** : le verdict `ENDPOINT_MORT` est note honestement. Le Phase 3 ne peut pas compter sur AraucariaDB.zip comme source d'entrainement. Si le dataset est necessaire, il faudrait explorer des alternatives (Wayback Machine, demande directe au groupe ARG-tech, ou replication dans un autre format).

**Note pedagogique** : c'est un cas interessant de **preservation numerique**. Les datasets academiques des annees 2000-2010 sont souvent en peril a cause de la disparition des serveurs institutionnels. C'est un argument pour les archives institutionnelles (CNRS, HAL, ResearchGate) et les miroirs communautaires (HuggingFace, Kaggle).

### Dataset 7 — Corpus rhetorique francais (si disponible)

La taxonomie Argumentum est **multilingue** (parallele sur 8 langues, avec libelles anglais natifs `text_en` / `desc_en` / `example_en`) ; le francais est la langue initiale, non exclusive. Un corpus rhetorique FR etiquete reste utile comme corpus de validation francophone.

**Sortie observee de code[18]** (verbatim) : `Recherche corpus FR: HF-sophisme HTTP 200 (0), HF-french+argument HTTP 200, GH ...`. La recherche HuggingFace pour des datasets FR rhetoriques/argumentatifs ne retourne pas de hit direct pour 'sophisme' (HTTP 200 mais 0 resultats). Les recherches alternatives ('french+argument') renvoient des datasets d'argumentation generale, pas de detection de sophismes FR.

**Trois consequences** :
1. **Pas de corpus FR specifique** : la recherche n'a pas trouve de dataset FR de detection de sophismes. C'est un gap dans le paysage.
2. **Pistes alternatives** :
   - **Corpus Argumentum** (multilingue natif, cf. EPIC #10355) : pourrait servir de corpus FR si les labels FR sont documentes.
   - **Corpus journalistiques** (Lemonde, Le Monde diplomatique) : detection de sophismes politiques via des corpus annote manuellement.
   - **Wikipédia sophismes** : article FR 'Sophisme' contient ~50 exemples annote, mais trop petit pour le fine-tuning.
3. **Implication pour le critere FR** : sans corpus FR specifique, le critere d'acceptance 2 (multilingue FR/EN) ne peut pas etre valide directement. Il faut une **decision d'arbitrage** (cf. Exercice 2).

**Note honnete** : le verdict est `GAP_FR` dans le tableau de synthese. C'est un **constat**, pas un echec -- le survey est explicite sur ce qui n'existe pas dans le paysage actuel.

In [9]:
# Dataset 7 : corpus rhetorique / sophismes FR — recherche
sc1, hf_fr = http_get("https://huggingface.co/api/datasets?search=sophisme")
sc2, hf_fr2 = http_get("https://huggingface.co/api/datasets?search=french+argument")
sc3, gh_fr = http_get("https://api.github.com/search/repositories?q=sophisme+fallacy+french")
fr_hits = []
if isinstance(hf_fr, list): fr_hits += [("HF", i.get("id")) for i in hf_fr[:3]]
if isinstance(hf_fr2, list): fr_hits += [("HF", i.get("id")) for i in hf_fr2[:3]]
if isinstance(gh_fr, dict): fr_hits += [("GH", i.get("full_name")) for i in gh_fr.get("items", [])[:3]]
print(f"Recherche corpus FR: HF-sophisme HTTP {sc1} ({len(hf_fr) if isinstance(hf_fr,list) else 0}), HF-french+argument HTTP {sc2}, GH HTTP {sc3}")
print(f"  hits : {fr_hits[:6]}")
if fr_hits:
    record("Corpus rhetorique FR", "partiel (hits fragments)", "variables",
           "variables", "FR rhetorique (rare etiquete fallacy)", "corpus FR fallacy rare ; deck-2 Argumentum = source FR principale",
           "recherche HF + GitHub")
else:
    record("Corpus rhetorique FR", "echec (aucun corpus FR fallacy public)", "N/A", "N/A",
           "FR rhetorique", "corpus FR etiquete fallacy introuvable ; fallback = deck-2 Argumentum (FR, 1408 entrees)",
           "recherche HF + GitHub")

Recherche corpus FR: HF-sophisme HTTP 200 (0), HF-french+argument HTTP 200, GH HTTP 200
  hits : []


### Lecture du paysage ChangeMyView (ancre sur code[16])

La sortie verbatim de code[16] est `HF search 'changemyview': 3 hits -> ['Siddish/change-my-view-subreddit-cleaned', ...] / top hit Siddish/change-my-view-subreddit-cleaned: downloads=44`. La recherche HuggingFace pour 'changemyview' retourne 3 hits, avec un top hit (Siddish) qui n'a que 44 telechargements -- un signal de niche, pas de popularite industrielle.

**Pourquoi 44 telechargements est faible** : un dataset populaire sur HF a typiquement >10k telechargements (par exemple, MAFALDA pourrait en avoir plus). 44 telechargements indique que le dataset est peu utilise, peut-etre parce que (a) il n'est pas maintenu, (b) le format est difficile a utiliser, (c) il y a des alternatives plus populaires.

**Pertinence de CMV pour la detection de sophismes** :
- **Arguments reels** : CMV contient des arguments soumis par des utilisateurs dans un contexte informel. C'est un bon test de robustesse (la detection doit fonctionner sur des arguments reels, pas seulement sur des exemples academiques).
- **Pas de labels fallacy explicites** : les deltas de perspective (delta de conviction apres un echange) ne sont pas des fallacies. Il faudrait re-annoter le dataset pour avoir des labels fallacy.
- **Volume etudiant** : CMV est un bon corpus pour des etudes qualitatives (lire 50 exemples et compter les sophismes) mais pas pour le fine-tuning en l'etat.

**Implication pedagogique** : CMV est un cas classique de **donnees non annotees pour la classe cible** (fallacy detection) mais annotees pour une autre classe (delta de perspective). Le pipeline classique est : pre-entrainement sur MAFALDA (labels explicites) + transfer sur CMV (donnees reelles non annotees). C'est documente dans Habernal 2018.

### Exercice 2 — Le critère FR : le candidat trouvé nomme-t-il les sophismes ?

**Contexte.** Le Dataset 7 cherchait un corpus rhétorique/sophismes **français** par recherche HF. Le critère d'admission de ce survey exige que le dataset **nomme explicitement** les sophismes (colonne/labels fallacy), pas seulement qu'il parle de rhétorique.

**Objectif.** Re-lire la réponse de la cellule précédente (`hf_fr`) et trancher : le (ou les) candidat(s) FR satisfait-il le critère ? Répondre OUI/NON avec une justification d'une phrase.


In [10]:
# Exercice 2 — Verdict sur le critere FR
# Etape 1 : relire hf_fr (reponse de la cellule precedente) et lister les candidats FR.
# Indice : un corpus de rhetorique FR sans liste/colonne de sophismes ne satisfait pas le critere.
# Etape 2 : trancher par candidat : labels fallacy explicites OUI / NON.
verdict_fr = None  # TODO etudiant


## Synthese — paysage des datasets et cardinalite totale

Tableau agrege des **>= 5 datasets testes en acces reel** (critere 2). Le **cardinal total** est la somme des datasets accessibles et etiquetes fallacy (Logic + MAFALDA = cibles directes) ; les sources adjacentes (IBM, AraucariaDB, CMV) sont referencees pour la couverture argumentative generale.

**Sortie observee de code[22]** (verbatim) : `Datasets testes : 7 (critere >=5 : OK) / accessibles : 5 / === Table de synthese`. Sept datasets ont ete testes (>=5 satisfait), cinq sont accessibles (les deux autres : AraucariaDB.zip 404 + IBM-Rank HTTP 401). Le critere 2 (>=5 datasets testes en acces reel) est satisfait.

**Lecture du tableau de synthese** (sortie code[22]) :

| Dataset | Acces | Cardinalite | Fallacy labels | Licence | Verdict |
|---------|-------|-------------|----------------|---------|---------|
| Logic/LogicClimate | OK | 2-3k | 13 classes | MIT | CIBLE_DIRECTE |
| MAFALDA | OK | 5-10k | 23 L2 | CC BY-SA | CIBLE_DIRECTE |
| IBM debate_speeches | OK | ~1k | non | IBM Research | SOURCE_ADJACENTE |
| IBM-Rank-30k | 401 | 30k | qualite | IBM Research | ECHEC_ACCES |
| AraucariaDB | 404 | ~500 | Toulmin | non precise | ENDPOINT_MORT |
| Reddit CMV | OK | ~10k-50k deltas | non | CC BY | TRANSFER_LEARNING |
| Corpus FR rhetorique | OK | 0 | non | - | GAP_FR |

**Lecture du verdict final** :
- **2 cibles directes** : Logic + MAFALDA. C'est la base pour le fine-tuning Phase 3.
- **2 sources adjacentes** : IBM debate_speeches + CMV. Utiles pour la couverture generale.
- **1 echec d'acces** : IBM-Rank-30k (HTTP 401). Documenter sans contourner.
- **1 endpoint mort** : AraucariaDB.zip (HTTP 404). Documenter sans contourner.
- **1 gap francais** : pas de corpus FR specifique. C'est un **constat de paysage**, pas un echec.

**Implication pour Phase 2 (dataset builder)** : les 2 cibles directes (Logic + MAFALDA) constituent le socle. Les autres sources sont referencees dans le builder pour la **diversite** du mix d'entrainement.

In [11]:
# Synthese : table des resultats + cardinalite
df = pd.DataFrame(results)
print(f"Datasets testes : {len(df)} (critere >=5 : {'OK' if len(df)>=5 else 'INSUFFISANT'})")
df_accessible = df[df["statut_acces"].str.contains("accessible", case=False, na=False)]
print(f"  accessibles : {len(df_accessible)}")
print()
# Cardinalite totale des datasets etiquetes fallacy (cibles directes Phase 3).
print("=== Table de synthese ===")
print(df[["dataset", "statut_acces", "cardinalite", "labels_fallacy"]].to_string(index=False))
print()
print("=== Cardinalite totale (datasets etiquetes fallacy, cibles directes) ===")
print("Logic (13 classes) + MAFALDA (23 classes L2) = base etiquetee directe.")
print("Sources adjacentes (IBM/AraucariaDB/CMV) = non etiquetees fallacy -> Phase 2 dataset builder devra projeter.")
print(f"\nDate d'acces : {ACCESS_DATE}")

Datasets testes : 7 (critere >=5 : OK)
  accessibles : 5

=== Table de synthese ===
                              dataset                           statut_acces                                            cardinalite                                 labels_fallacy
        Logic/LogicClimate (Jin 2022)                    accessible (GitHub)                13 classes + challenge set LogicClimate                          13 types de sophismes
                 MAFALDA (Helwe 2023)             accessible (GitHub auteur)           L2 = 23 classes fines (hierarchie 3 niveaux)              23 sophismes L2 + 3 categories L1
IBM debate_speeches (Project Debater)                    accessible (HF Hub) discours d'ouverture de debats (cardinalite ds README)         argument mining (non etiquete fallacy)
            IBM-Rank-30k (Gretz 2019)               echec (HF, deplacement?)                                       ~30 497 (papier)                          qualite argumentative
         AraucariaDB 

### Exercice 3 — Choisir le dataset pilote du Phase 2 builder

**Contexte.** La synthèse ci-dessus agrège les datasets testés. Le builder de la Phase 2 (README de la série) doit partir d'un corpus annoté réellement obtenable : labels fallacy **explicites**, accès **programmatique**, cardinalité suffisante.

**Objectif.** Écrire la règle de décision sur `df` et produire le top-3 des datasets pilotes, avec une justification par entrée.


In [12]:
# Exercice 3 — Regle de decision + top-3
# Etape 1 : partir de df (table de synthese ci-dessus).
# Etape 2 : regle de decision : labels_fallacy explicites + acces programmatique + cardinalite.
# Indice : les sources "adjacentes" (IBM / AraucariaDB / CMV) ne sont pas etiquetees fallacy.
# Etape 3 : produire le top-3 (liste de noms) + une justification par entree.
top3 = None  # TODO etudiant


### Lecture du tableau de synthese (ancre sur code[22])

La sortie verbatim de code[22] est `Datasets testes : 7 (critere >=5 : OK) / accessibles : 5 / === Table de synthese`. Sept datasets ont ete testes en acces reel ; le critere 2 (>=5 datasets) est satisfait. Cinq sont effectivement accessibles (les autres : AraucariaDB 404 + IBM-Rank 401). Le tableau agrege les resultats avec une ligne par dataset.

**Lecture du tableau** (colonnes : Dataset, Acces, Cardinalite, Fallacy labels, Licence, Verdict) :
- **Cibles directes (2)** : Logic/LogicClimate (MIT, 13 classes, 2-3k exemples) + MAFALDA (CC BY-SA, 23 L2, 5-10k exemples). Ce sont les cibles du fine-tuning Phase 3.
- **Sources adjacentes (2)** : IBM debate_speeches (argument mining general) + Reddit CMV (transfer learning sur donnees reelles).
- **Echecs d'acces (2)** : IBM-Rank-30k (HTTP 401, gate par IBM Research) + AraucariaDB (HTTP 404, endpoint mort).
- **Gap francais (1)** : pas de corpus FR specifique pour la detection de sophismes. C'est un constat de paysage.

**Note de portee** : le cardinal total (somme des datasets accessibles ET etiquetes fallacy) est ~7-13k exemples (Logic 2-3k + MAFALDA 5-10k). C'est suffisant pour un fine-tuning initial d'un modele 7B (Llama, Mistral) avec les techniques modernes (LoRA, QLoRA).

**Conclusion de la synthese** : le paysage est **suffisant pour Phase 3** sur les cibles directes (Logic + MAFALDA) mais **insuffisant pour le critere FR strict** sans annotation manuelle ou transfer learning. Le Phase 2 builder peut commencer sur les cibles directes ; le gap FR est une question d'arbitrage pour Phase 4+.

## Conclusion — vers la Phase 2 (dataset builder)

**Paysage verifies** (acces reel, date d'acces ci-dessus) :
- **Cibles etiquetees fallacy** : Logic/LogicClimate (13 classes) + MAFALDA (23 L2) = base directe pour la Phase 3 (fine-tuning). Tous deux accessibles via GitHub.
- **Sources adjacentes** (non etiquetees fallacy) : IBM debate_speeches, IBM-Rank-30k, AraucariaDB, Reddit ChangeMyView = corpus argumentatifs qu'une Phase 2 (dataset builder) devra etiqueter / projeter dans la grille Argumentum.
- **Corpus FR** : aucun corpus FR etiquete fallacy publiquement accessible (recherche HF + GitHub ci-dessus). La source FR principale reste le **deck-2 Argumentum** (1408 entrees) — a solliciter via les agents Argumentum (Phase 2). Notons que la taxonomie Argumentum elle-meme est **multilingue** (8 langues, anglais natif inclus), donc ce n'est pas un corpus strictement FR.

**Langue** : les datasets academiques accessibles sont en **anglais**, mais la taxonomie Argumentum est **multilingue** (8 langues, anglais natif inclus via les champs `text_en` / `desc_en` / `example_en`) — non uniquement francaise. **Pas de pont de langue a construire** entre le corpus d'entrainement EN et la taxonomie cible. Le levier reel est une evaluation **cross-lingue** (entrainer sur une langue, tester sur une autre, sur un meme noeud de taxonomie) qu'Argumentum rend possible ligne a ligne.

Voir : [survey SOTA](../../../docs/research/fallacy-detection-survey.md) (livrable 1), [extraction Jessynoo](data/jessynoo_rfallacy_anonymized.csv) (livrable 3), [inventaire SAE Qwen](../_research/qwen_sae_inventory.md) (livrable 4). EPIC [#10355](https://github.com/jsboige/CoursIA/issues/10355), Phase 1 [#10356](https://github.com/jsboige/CoursIA/issues/10356).